# QLoRA Fine-Tuning meta-llama/Llama-3.2-3B

In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1
!pip install -q torch transformers accelerate peft datasets wandb matplotlib
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

In [ ]:
import os
import re
from datetime import datetime
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed, BitsAndBytesConfig,GenerationConfig
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
import wandb
from util import evaluate, Tester

hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

set_seed(42)

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "Xander-K"
LITE_MODE = True
DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

MINIMAL_MODE = False
if MINIMAL_MODE:
    N_TRAIN = 500
    N_VAL = 100
    EVAL_SIZE = 150
    EPOCHS = 1
    BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 2
    MAX_SEQUENCE_LENGTH = 128
    LORA_R = 32
    VAL_SIZE = 100
    LOG_STEPS = 5
    SAVE_STEPS = 50
    SAVE_TOTAL_LIMIT = 2
else:
    N_TRAIN = None
    N_VAL = None
    EVAL_SIZE = 200
    EPOCHS = 1 if LITE_MODE else 3
    BATCH_SIZE = 32 if LITE_MODE else 256
    GRADIENT_ACCUMULATION_STEPS = 1
    MAX_SEQUENCE_LENGTH = 128
    LORA_R = 32 if LITE_MODE else 256
    VAL_SIZE = 500 if LITE_MODE else 1000
    LOG_STEPS = 5 if LITE_MODE else 10
    SAVE_STEPS = 100 if LITE_MODE else 200
    SAVE_TOTAL_LIMIT = 10

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
    RUN_NAME += "-lite"
if MINIMAL_MODE:
    RUN_NAME += "-minimal"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

LORA_ALPHA = LORA_R * 2
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_DROPOUT = 0.1
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = "cosine"
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

LOG_TO_WANDB = True
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"
if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

print(f"MINIMAL_MODE={MINIMAL_MODE}, N_TRAIN={N_TRAIN}, N_VAL={N_VAL}")
print(f"BASE_MODEL={BASE_MODEL}, OUTPUT={PROJECT_RUN_NAME}")

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset["train"]
val = dataset["val"].select(range(VAL_SIZE))
test = dataset["test"]

if MINIMAL_MODE and N_TRAIN:
    train = train.select(range(min(N_TRAIN, len(train))))
if MINIMAL_MODE and N_VAL:
    val = val.select(range(min(N_VAL, len(val))))

def add_text(row):
    return {"text": row["prompt"] + row["completion"]}

train = train.map(add_text)
val = val.map(add_text)
print(f"Train {len(train)}, val {len(val)}, test {len(test)}")

In [ ]:
if use_bf16:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
)

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters,
)
print("Trainer ready.")

In [ ]:
fine_tuning.train()
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to hub: {PROJECT_RUN_NAME}")
if LOG_TO_WANDB:
    wandb.finish()

In [ ]:
fine_tuned = fine_tuning.model
fine_tuned.eval()

def extract_price_from_output(text):
    if "Price is $" in text:
        part = text.split("Price is $")[-1].strip().replace(",", "")
        m = re.search(r"[-+]?\d*\.?\d+", part)
        return float(m.group()) if m else 0.0
    m = re.search(r"[-+]?\d*\.?\d+", text)
    return float(m.group()) if m else 0.0

def model_predict(datapoint):
    prompt = datapoint["prompt"]
    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned.device)
    with torch.no_grad():
        out = fine_tuned.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    price = extract_price_from_output(decoded)
    return f"{price:.2f}"

In [ ]:

fine_tuned = fine_tuning.model
fine_tuned.eval()

# Generation config with only valid flags (avoids temperature/top_p warning)
gen_config = GenerationConfig(max_new_tokens=12, do_sample=False, pad_token_id=tokenizer.eos_token_id)

def extract_price_from_output(text):
    if "Price is $" in text:
        part = text.split("Price is $")[-1].strip().replace(",", "")
        m = re.search(r"[-+]?\d*\.?\d+", part)
        return float(m.group()) if m else 0.0
    m = re.search(r"[-+]?\d*\.?\d+", text)
    return float(m.group()) if m else 0.0

def model_predict(datapoint):
    prompt = datapoint["prompt"]
    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned.device)
    with torch.no_grad():
        device = next(fine_tuned.parameters()).device
        if device.type == "cuda":
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                out = fine_tuned.generate(**inputs, generation_config=gen_config)
        else:
            out = fine_tuned.generate(**inputs, generation_config=gen_config)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    price = extract_price_from_output(decoded)
    return f"{price:.2f}"

In [ ]:
evaluate(model_predict, test, size=EVAL_SIZE)